# A-Share Multifactor Research

This notebook loads cached data, runs IC analysis, and visualizes quantile backtest results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from a_share_multifactor.backtest import run_pipeline
from a_share_multifactor.config import load_config

CONFIG_PATH = Path("../configs/default.yaml")
OUTPUT_DIR = Path("../outputs/latest")
config = load_config(CONFIG_PATH)

In [ ]:
# Run pipeline if outputs are missing
if not (OUTPUT_DIR / "ic_summary.csv").exists():
    run_pipeline(config, data_dir=Path("../data"))

ic_summary = pd.read_csv(OUTPUT_DIR / "ic_summary.csv")
quantile_returns = pd.read_csv(OUTPUT_DIR / "quantile_returns.csv", index_col=0, parse_dates=True)
cumulative_returns = pd.read_csv(OUTPUT_DIR / "cumulative_returns.csv", index_col=0, parse_dates=True)
ic_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ic_summary["factor"], ic_summary["ir"])
ax.set_title("Information Ratio by Factor")
ax.set_ylabel("IR")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for col in cumulative_returns.columns:
    ax.plot(cumulative_returns.index, cumulative_returns[col], label=col)
ax.set_title("Quantile Portfolio Cumulative Returns")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IC decay heatmap
ic_decay = pd.read_csv(OUTPUT_DIR / "ic_decay.csv")
pivot = ic_decay.pivot(index="factor", columns="horizon_days", values="mean_ic")

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title("IC Decay Heatmap")
ax.set_xlabel("Forward Return Horizon (days)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Factor correlation matrix (latest cross-section snapshot)
factor_cols = [c for c in config.factors if c in ic_summary["factor"].tolist()]
ic_files = list((OUTPUT_DIR / "ic_series").glob("ic_series_*.csv")) if (OUTPUT_DIR / "ic_series").exists() else []

# Load panel for correlation if pipeline was run
from a_share_multifactor.data_loader import build_dataset
from a_share_multifactor.preprocess import prepare_factor_panel

panel = prepare_factor_panel(config, build_dataset(config, data_dir=Path("../data")))
available = [c for c in config.factors if c in panel.columns]
latest_date = panel["date"].max()
snapshot = panel[panel["date"] == latest_date][available].dropna()
if len(snapshot.columns) >= 2:
    corr = snapshot.corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.index)))
    ax.set_yticklabels(corr.index)
    ax.set_title(f"Factor Correlation ({latest_date.date()})")
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

In [ ]:
if (OUTPUT_DIR / "long_short.csv").exists():
    long_short = pd.read_csv(OUTPUT_DIR / "long_short.csv", index_col=0, parse_dates=True)
    long_short_cum = (1 + long_short.iloc[:, 0]).cumprod()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(long_short_cum.index, long_short_cum.values)
    ax.set_title("Long-Short (Top Quantile - Bottom Quantile)")
    plt.tight_layout()
    plt.show()